In [7]:
import json
import numpy as np
import pandas as pd
import librosa
import torch
import torchaudio
from pathlib import Path
from tqdm import tqdm

In [8]:
CONFIG = {
    # Paths
    'data_root': Path(r'C:\Users\SIVA M\Documents\CSE 575 Statistcal Machine Learning\Project\Dataset\V5\Data'),
    'output_root': Path('processed_mels'),  # Same as before
    
    # Audio source
    'audio_source': 'all',  # Same as preprocessing
    
    # Audio parameters
    'target_sr': 16000,
    
    # Mel-spectrogram parameters (MUST match preprocessing)
    'n_fft': 1024,
    'hop_length': 256,
    'n_mels': 128,
    'fmin': 20,
    'fmax': None,
    
    # Normal segment extraction
    'normal_segment_duration': 10.0,  # Extract 10-second normal segments
    'min_gap_duration': 15.0,         # Only use gaps longer than 15 seconds
    'segments_per_gap': 1,            # How many segments to extract per gap
    
    # Target: match number of apnea segments
    'max_normal_segments': None,  # None = auto-balance with apnea count
    
    # GPU settings
    'use_gpu': True,
    'batch_size': 16,
}

# Setup device
device = torch.device('cuda' if CONFIG['use_gpu'] and torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

Device: cuda


In [9]:
class MelTransform:
    """GPU-accelerated mel-spectrogram computation"""
    
    def __init__(self, sr, n_fft, hop_length, n_mels, fmin, fmax, device):
        self.device = device
        
        self.mel_transform = torchaudio.transforms.MelSpectrogram(
            sample_rate=sr,
            n_fft=n_fft,
            hop_length=hop_length,
            n_mels=n_mels,
            f_min=fmin,
            f_max=fmax if fmax else sr // 2,
        ).to(device)
        
        self.amplitude_to_db = torchaudio.transforms.AmplitudeToDB(
            stype='power'
        ).to(device)
    
    def __call__(self, waveform):
        if waveform.dim() == 1:
            waveform = waveform.unsqueeze(0)
        mel_spec = self.mel_transform(waveform)
        mel_db = self.amplitude_to_db(mel_spec)
        return mel_db

def normalize_mel_gpu(mel):
    """Normalize mel-spectrogram on GPU"""
    if mel.dim() == 2:
        mel = mel.unsqueeze(0)
    
    mel = mel.contiguous()
    batch_size = mel.shape[0]
    mel_flat = mel.reshape(batch_size, -1)
    mel_min = mel_flat.min(dim=1, keepdim=True)[0].unsqueeze(2)
    mel_max = mel_flat.max(dim=1, keepdim=True)[0].unsqueeze(2)
    
    range_vals = mel_max - mel_min
    range_vals[range_vals == 0] = 1.0
    
    mel_normalized = (mel - mel_min) / range_vals
    return mel_normalized

In [10]:
def find_normal_segments(annotation_path, audio_duration, config):
    """
    Find time periods where there are NO apnea events.
    
    Args:
        annotation_path: Path to annotation JSON
        audio_duration: Total audio duration in seconds
        config: Configuration dict
    
    Returns:
        List of (start_sec, end_sec) tuples for normal segments
    """
    # Load annotations
    with open(annotation_path, 'r') as f:
        data = json.load(f)
    
    record_start = float(data.get("record_start", 0))
    events = data.get("events", [])
    
    # Get all apnea event intervals
    apnea_intervals = []
    for ev in events:
        ev_start = float(ev.get("evnet_start", ev.get("event_start", 0)))
        ev_dur = float(ev.get("event_duration", 0))
        
        start_sec = max(0.0, ev_start - record_start)
        end_sec = start_sec + ev_dur
        
        apnea_intervals.append((start_sec, end_sec))
    
    # Sort by start time
    apnea_intervals.sort(key=lambda x: x[0])
    
    # Find gaps between apnea events
    normal_gaps = []
    
    # Gap before first event
    if apnea_intervals and apnea_intervals[0][0] > config['min_gap_duration']:
        normal_gaps.append((0, apnea_intervals[0][0]))
    
    # Gaps between events
    for i in range(len(apnea_intervals) - 1):
        gap_start = apnea_intervals[i][1]  # End of current event
        gap_end = apnea_intervals[i+1][0]  # Start of next event
        gap_duration = gap_end - gap_start
        
        if gap_duration >= config['min_gap_duration']:
            normal_gaps.append((gap_start, gap_end))
    
    # Gap after last event
    if apnea_intervals:
        last_end = apnea_intervals[-1][1]
        if audio_duration - last_end >= config['min_gap_duration']:
            normal_gaps.append((last_end, audio_duration))
    
    # Extract segments from each gap
    normal_segments = []
    segment_dur = config['normal_segment_duration']
    segments_per_gap = config['segments_per_gap']
    
    for gap_start, gap_end in normal_gaps:
        gap_duration = gap_end - gap_start
        
        # How many segments can fit?
        max_segments = int(gap_duration // segment_dur)
        n_segments = min(segments_per_gap, max_segments)
        
        # Evenly space segments in the gap
        if n_segments > 0:
            for i in range(n_segments):
                # Center segments in the gap
                offset = (gap_duration - n_segments * segment_dur) / 2
                seg_start = gap_start + offset + i * segment_dur
                seg_end = seg_start + segment_dur
                
                if seg_end <= gap_end:
                    normal_segments.append((seg_start, seg_end))
    
    return normal_segments

def find_audio_file(patient_folder, patient_id, audio_source):
    """Find audio file(s) for a patient"""
    audio_files = []
    
    if audio_source == 'all':
        audio_files = list(patient_folder.glob(f"{patient_id}_*.wav"))
        return audio_files if audio_files else None
    elif audio_source == 'both_recorders':
        for recorder in ['recorder_1', 'recorder_2', 'recorder']:
            audio_path = patient_folder / f"{patient_id}_{recorder}.wav"
            if audio_path.exists():
                audio_files.append(audio_path)
        return audio_files if audio_files else None
    else:
        audio_path = patient_folder / f"{patient_id}_{audio_source}.wav"
        if audio_path.exists():
            return [audio_path]
        return None

In [11]:
def generate_normal_segments(config):
    """Generate normal breathing segments"""
    
    print("\n" + "=" * 70)
    print("GENERATING NORMAL SEGMENTS")
    print("=" * 70)
    print(f"Device: {device}")
    print(f"Audio source: {config['audio_source']}")
    print(f"Normal segment duration: {config['normal_segment_duration']}s")
    print(f"Min gap duration: {config['min_gap_duration']}s")
    print("=" * 70)
    
    # Setup
    data_root = config['data_root']
    output_root = config['output_root']
    seg_root = output_root / 'mel_segments'
    
    # Initialize mel transform
    mel_transform = MelTransform(
        sr=config['target_sr'],
        n_fft=config['n_fft'],
        hop_length=config['hop_length'],
        n_mels=config['n_mels'],
        fmin=config['fmin'],
        fmax=config['fmax'],
        device=device
    )
    
    # Load existing metadata to see how many apnea segments we have
    existing_meta = pd.read_csv(output_root / 'all_metadata.csv')
    n_apnea = len(existing_meta)
    
    print(f"\nExisting apnea segments: {n_apnea}")
    print(f"Target normal segments: {n_apnea} (to balance classes)")
    
    # Find patient folders
    patient_folders = sorted([p for p in data_root.iterdir() if p.is_dir()])
    
    all_metadata = []
    total_normal_count = 0
    target_normal = config['max_normal_segments'] if config['max_normal_segments'] else n_apnea
    
    # Process each patient
    for folder in tqdm(patient_folders, desc="Processing patients"):
        patient_id = folder.name
        
        # Check if we've generated enough normal segments
        if total_normal_count >= target_normal:
            break
        
        # Find annotation
        ann_path = folder / f"{patient_id}_annotation.json"
        if not ann_path.exists():
            continue
        
        # Find audio files
        audio_files = find_audio_file(folder, patient_id, config['audio_source'])
        if not audio_files:
            continue
        
        # Process each audio file
        for audio_path in audio_files:
            if total_normal_count >= target_normal:
                break
            
            audio_name = audio_path.stem
            
            # Load audio to get duration
            try:
                y, sr = librosa.load(str(audio_path), sr=config['target_sr'])
                audio_duration = len(y) / sr
            except:
                continue
            
            # Find normal segments (gaps between apnea events)
            normal_segments = find_normal_segments(ann_path, audio_duration, config)
            
            if not normal_segments:
                continue
            
            # Convert to torch tensor on GPU
            audio_tensor = torch.from_numpy(y).to(device)
            
            # Process each normal segment
            for seg_idx, (start_sec, end_sec) in enumerate(normal_segments):
                if total_normal_count >= target_normal:
                    break
                
                # Extract segment
                start_sample = int(start_sec * sr)
                end_sample = int(end_sec * sr)
                segment = audio_tensor[start_sample:end_sample]
                
                if len(segment) == 0:
                    continue
                
                # Compute mel-spectrogram on GPU
                segment_batch = segment.unsqueeze(0)  # (1, samples)
                mel = mel_transform(segment_batch)
                mel = normalize_mel_gpu(mel)
                
                # Move to CPU
                mel_cpu = mel.squeeze(0).cpu().numpy().astype(np.float32)
                
                # Save
                filename = f"{patient_id}_{audio_name}_normal_{seg_idx:05d}.npy"
                save_path = seg_root / filename
                np.save(save_path, mel_cpu)
                
                # Record metadata
                all_metadata.append({
                    'file': str(save_path),
                    'label': 'normal',
                    'event_type': 'normal',
                    'patient_id': patient_id,
                    'audio_source': audio_name,
                    'audio_file': str(audio_path),
                    'duration_sec': end_sec - start_sec,
                    'mel_shape': f"{mel_cpu.shape[0]}x{mel_cpu.shape[1]}",
                })
                
                total_normal_count += 1
            
            # Clear GPU cache
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
    
    # Save normal segments metadata
    if not all_metadata:
        print("\n No normal segments generated!")
        return None
    
    normal_df = pd.DataFrame(all_metadata)
    normal_meta_path = output_root / 'normal_metadata.csv'
    normal_df.to_csv(normal_meta_path, index=False)
    
    # Combine with existing apnea segments
    combined_df = pd.concat([existing_meta, normal_df], ignore_index=True)
    combined_meta_path = output_root / 'all_metadata.csv'
    combined_df.to_csv(combined_meta_path, index=False)
    
    # Print summary
    print("\n" + "=" * 70)
    print(" NORMAL SEGMENTS GENERATED")
    print("=" * 70)
    print(f"\nNormal segments generated: {len(normal_df)}")
    print(f"Total segments (apnea + normal): {len(combined_df)}")
    print(f"\nLabel distribution:")
    print(combined_df['label'].value_counts())
    print(f"\nMetadata saved to: {combined_meta_path}")
    print("=" * 70)
    
    return combined_meta_path

In [12]:
if __name__ == "__main__":
    import time
    
    print("\n  IMPORTANT: This will generate 'normal' class segments")
    print("These are extracted from gaps between apnea events.")
    print("\nThis will:")
    print("  1. Generate ~24,000 normal segments (to match apnea count)")
    print("  2. Update all_metadata.csv with combined data")
    print("  3. Take ~10-15 minutes with GPU")
    
    response = input("\nContinue? (yes/no): ").strip().lower()
    
    if response != 'yes':
        print("Cancelled.")
        exit()
    
    start_time = time.time()
    metadata_path = generate_normal_segments(CONFIG)
    elapsed_time = time.time() - start_time
    
    if metadata_path:
        print(f"\n Processing time: {elapsed_time/60:.1f} minutes")
        print("\n SUCCESS!")
        print("\n Next step:")
        print("  python optimized_resnet_osa.py")
    else:
        print("\n FAILED")


  IMPORTANT: This will generate 'normal' class segments
These are extracted from gaps between apnea events.

This will:
  1. Generate ~24,000 normal segments (to match apnea count)
  2. Update all_metadata.csv with combined data
  3. Take ~10-15 minutes with GPU

GENERATING NORMAL SEGMENTS
Device: cuda
Audio source: all
Normal segment duration: 10.0s
Min gap duration: 15.0s

Existing apnea segments: 24255
Target normal segments: 24255 (to balance classes)


Processing patients: 100%|██████████| 50/50 [11:01<00:00, 13.23s/it]



 NORMAL SEGMENTS GENERATED

Normal segments generated: 20589
Total segments (apnea + normal): 44844

Label distribution:
label
apnea     24255
normal    20589
Name: count, dtype: int64

Metadata saved to: processed_mels\all_metadata.csv

 Processing time: 11.1 minutes

 SUCCESS!

 Next step:
  python optimized_resnet_osa.py
